# Episodic Memory in Agentic AI: A Practical Demonstration

**Episodic memory = a persistent record of what happened in past sessions** (what was discussed, decided, produced, and what the outcome was).

It is **not** working memory. Working memory is the context window and disappears when a session ends. Episodic memory is **stored outside the LLM** (here, in a file), can grow without limit, and **only the episodes we retrieve for the current need are put back into the context window**.

In this notebook we demonstrate, with **real LLM calls only** (Groq `openai/gpt-oss-20b` + a real web-search tool):

| Section | What we show |
|---|---|
| 1 | The problem: a returning user says "continue what we were doing" and the agent has no idea |
| 2 | Writing an episode: summary, key events and outcomes (not the raw transcript) |
| 3 | Building a small history of past sessions |
| 4 | Retrieval type 1: chronological ("last N sessions") |
| 5 | Retrieval type 2: semantic ("sessions similar to this task") |
| 6 | Using stored key events and outcomes |
| 7 | Disadvantage: summaries lose detail |
| 8 | Final comparison with the real numbers from this notebook |

**How to read this notebook honestly:**
- We can't wait a week between sessions, so sessions run minutes apart. Timestamps are real, working memory is genuinely emptied between sessions, and the episode file is the only thing that carries over. The mechanism is identical to a week later.
- The **user's messages in each session are scripted by us** (like a test script). **Every assistant reply, search, episode summary and answer is a real LLM call.**
- LLM outputs vary slightly between runs. If Groq's free tier returns a rate-limit error, wait a few seconds and re-run that cell.

# 0. Setup

In [ ]:
%pip install -q langchain-groq langgraph langchain-community duckduckgo-search ddgs sentence-transformers

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Bring the API key from Colab secrets
my_api_key = userdata.get('agentic_ai')

# Plain LLM (no tools). We use it to write episode summaries.
llm = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0,
    api_key=my_api_key,
    max_retries=6   # retry automatically if we hit Groq's free-tier rate limit
)

## 0.1 The agent (same pattern as before)

An assistant node and a tool node in a loop. Its `State["messages"]` is the **working memory** of one session.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_community.tools import DuckDuckGoSearchRun

# ---- Real web-search tool ----
search = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Search the web with latest information and give the result"""
    # Search backends sometimes fail with temporary network errors (e.g. DNS), so retry a few times
    for attempt in range(3):
        try:
            # Keep only the first 800 characters so we stay inside Groq's free-tier token limits
            return search.run(query)[:800]
        except Exception as e:
            print(f"Search attempt {attempt + 1} failed: {str(e)[:80]}")
            time.sleep(2)
    # Tell the LLM the search failed instead of crashing the whole agent
    return "Web search is temporarily unavailable."

tools = [web_search]
llm_with_tools = llm.bind_tools(tools)   # LLM that is allowed to call the tool

# ---- State = working memory of the current session ----
class State(TypedDict):
    messages: Annotated[list, add_messages]

def assistant_node(state: State):
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response]}

def should_continue(state: State) -> Literal['tools', END]:
    return 'tools' if state['messages'][-1].tool_calls else END

builder = StateGraph(State)
builder.add_node('assistant', assistant_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'assistant')
builder.add_conditional_edges('assistant', should_continue)
builder.add_edge('tools', 'assistant')
graph = builder.compile()

## 0.2 Helpers

- `run_session` plays one whole session: our scripted user messages, the agent's real replies. Every session starts with an **empty working memory** unless we pass one in.
- `ask` asks a question with whatever memory we give it, prints the answer, and records the tokens sent and whether the answer contains the fact we expect. (The keyword check is only a quick signal. We always print the answer too.)

In [ ]:
results = {}   # we collect results here to compare everything at the end

def first_call_tokens(msgs):
    """Tokens sent in the FIRST LLM call after the latest user message
    (= the memory we carried in + the new question)."""
    last_human = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
    return msgs[last_human + 1].usage_metadata['input_tokens']

def to_text(messages):
    """Turn a list of messages into plain text (skips empty tool-call-only messages)."""
    return "\n".join(f"{m.type}: {m.content}" for m in messages if m.content)

def run_session(user_turns):
    """Run one session with an EMPTY working memory. User messages are scripted, replies are real."""
    memory = []
    for text in user_turns:
        result = graph.invoke({"messages": memory + [HumanMessage(text)]})
        memory = result["messages"]
        print("User :", text)
        print("Agent:", memory[-1].content[:300], "\n")
    return memory

def ask(question, memory, label, expected=None):
    """Ask a question with the given memory, print the answer and store the result."""
    result = graph.invoke({"messages": memory + [HumanMessage(question)]})
    msgs = result["messages"]
    answer = msgs[-1].content

    tokens = first_call_tokens(msgs)
    correct = (expected.lower() in answer.lower()) if expected else "n/a"
    results[label] = {"tokens sent": tokens, "answer has the expected fact": correct}

    print(f"[{label}]")
    print(f"Q: {question}")
    print(f"Tokens sent in first call: {tokens}")
    print(f"A: {answer}")
    if expected:
        print(f"Contains '{expected}'? {correct}")
    print()

# 1. The Problem: No Episodic Memory

## 1.1 Session 1: a real research conversation

The user is preparing a report, states a preference, gets help from the agent (with a real web search), approves an outline and gives a rule.

In [ ]:
ev_session = run_session([
    "I'm preparing a short research report on electric two-wheelers in India. Please keep your answers concise, I prefer short responses.",
    "Search the web for the latest news on electric two-wheeler sales in India and suggest a 3-part outline for the report.",
    "I approve the outline. One rule: do not include anything about charging infrastructure in the report.",
])

print("Tokens sent in the last call of this session (= the whole transcript):",
      ev_session[-1].usage_metadata['input_tokens'])

User : I'm preparing a short research report on electric two-wheelers in India. Please keep your answers concise, I prefer short responses.
Agent: Sure! What specific topics or data points would you like covered in your report? (e.g., market size, growth trends, key players, policy landscape, charging infrastructure, consumer adoption, challenges, etc.) 

Search attempt 1 failed: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/ap
Search attempt 2 failed: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/ap
User : Search the web for the latest news on electric two-wheeler sales in India and suggest a 3-part outline for the report.
Agent: **Three‑Part Outline for the Report**

| Section | Key Focus | Suggested Sub‑Points |
|---------|-----------|----------------------|
| **1. Market Landscape** | Current size, growth, and segmentation | • 2024 sales: 1.14 M units (59 % of all EVs) <br>• YoY growth ≈ 27 % (nearly a third increase) 

## 1.2 The user comes back (a new session): working memory is empty

The context window of Session 1 is gone. The user simply returns and asks to continue.

In [ ]:
# A brand-new session: nothing from Session 1 is sent to the model
result = graph.invoke({"messages": [HumanMessage("Can you continue what we were doing last time?")]})
print("A:", result["messages"][-1].content)

A: Sure thing! Could you remind me what we were working on last time? That way I can pick up right where we left off.


In [ ]:
# A more specific question about Session 1, also with no memory
ev_question = "Let's continue the electric two-wheeler report. What rule did I give about its content?"

ask(ev_question, [], "1. No memory | EV question", expected="charging infrastructure")

[1. No memory | EV question]
Q: Let's continue the electric two-wheeler report. What rule did I give about its content?
Tokens sent in first call: 143
A: I don’t see a rule you mentioned earlier about the report’s content. Could you remind me what rule you’d like me to follow?
Contains 'charging infrastructure'? False



**What to observe:** the agent has no idea what "last time" means. The LLM is stateless and the old context window no longer exists. We need a **persistent store of past sessions** that survives after the session ends.

# 2. Writing Episodes

At the **end of a session**, we turn it into an **episode record** and save it in a store outside the LLM.

We do **not** store the raw transcript (too long). An episode holds:
- `summary`: what was discussed, decided and produced
- `key_events`: preferences, approvals, rules, decisions, errors
- `outcomes`: what actions were taken and what happened
- `timestamp`: when it happened

The summary and the events are written by a real LLM call. We also store an **embedding** of the episode so we can search by meaning later.

## 2.1 The store and the write helpers

In [ ]:
from sentence_transformers import SentenceTransformer

# Small embedding model: turns text into a vector so we can compare meanings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

EPISODES_FILE = 'episodes.json'   # the persistent store (lives outside the LLM)

# Start with an empty store so the notebook can be re-run from the top
if os.path.exists(EPISODES_FILE):
    os.remove(EPISODES_FILE)

def load_episodes():
    """Read all episodes from the store."""
    if not os.path.exists(EPISODES_FILE):
        return []
    with open(EPISODES_FILE) as f:
        return json.load(f)

def episode_text(ep):
    """One piece of text that represents the whole episode."""
    events = "; ".join(str(x) for x in ep["key_events"])
    outcomes = "; ".join(str(x) for x in ep["outcomes"])
    return f"{ep['summary']} Key events: {events}. Outcomes: {outcomes}."

def make_episode(transcript):
    """Ask the LLM to compress a session transcript into an episode record."""
    prompt = (
        "Below is a conversation between a user and an AI assistant.\n"
        "Write a memory record of it as JSON with exactly these keys:\n"
        '"summary": 2-3 sentences on what was discussed, decided and produced\n'
        '"key_events": a list of short strings (user preferences, approvals, rules, decisions, errors)\n'
        '"outcomes": a list of short strings (actions taken and what happened as a result)\n'
        "Reply with the JSON only.\n\n" + to_text(transcript)
    )
    reply = llm.invoke(prompt).content
    # Cut out the JSON object from the reply and parse it
    return json.loads(reply[reply.index("{"): reply.rindex("}") + 1])

def save_episode(session_name, transcript):
    """Create an episode from a finished session and append it to the store."""
    episode = make_episode(transcript)
    episode["session"] = session_name
    episode["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    episode["embedding"] = embedder.encode(episode_text(episode)).tolist()

    episodes = load_episodes()
    episodes.append(episode)
    with open(EPISODES_FILE, "w") as f:
        json.dump(episodes, f)
    return episode

def show_episode(ep):
    """Print an episode without its (long) embedding vector."""
    print(json.dumps({k: v for k, v in ep.items() if k != "embedding"}, indent=2))

## 2.2 Save Session 1 as an episode

In [ ]:
ev_episode = save_episode("EV report", ev_session)

show_episode(ev_episode)
print("\nEpisodes in the store:", len(load_episodes()))

{
  "summary": "The user requested a concise outline for a research report on electric two\u2011wheelers in India. The assistant provided a three\u2011part outline, which the user approved and added a rule to exclude charging infrastructure. The assistant updated the outline accordingly.",
  "key_events": [
    "User prefers concise responses",
    "User asks for 3\u2011part outline",
    "Assistant provides outline",
    "User approves outline",
    "User adds rule: no charging infrastructure"
  ],
  "outcomes": [
    "Assistant delivered outline",
    "User approved outline",
    "Assistant revised outline to remove charging infrastructure references"
  ],
  "session": "EV report",
  "timestamp": "2026-11-19 15:26:56"
}

Episodes in the store: 1


**What to observe:** the whole conversation, including the search results, became a short record with the summary, the preference, the approval, the rule and the outcome. The store now lives in a file, completely outside the LLM and outside any context window.

# 3. Building a History of Past Sessions

We add two more sessions on **unrelated topics**, each saved as an episode right after it ends. Each session starts with an empty working memory.

## 3.1 Session 2: a data pipeline error (and how it was fixed)

In [ ]:
pipeline_session = run_session([
    "My data pipeline failed at step 3 with the error: KeyError: 'customer_id'. What could cause this?",
    "I found it. The column had been renamed to 'cust_id' in the source file. I fixed it by updating the column name in my code and the pipeline now runs fine.",
])

pipeline_episode = save_episode("Pipeline error", pipeline_session)
show_episode(pipeline_episode)

User : My data pipeline failed at step 3 with the error: KeyError: 'customer_id'. What could cause this?
Agent: A **`KeyError: 'customer_id'`** means that somewhere in step 3 your code is trying to look up a key called `customer_id` that isn’t present in the data structure you’re working with.  
Below is a quick‑fire checklist to help you pinpoint the root cause and fix it.

---

## 1. Identify the Data Struc 

User : I found it. The column had been renamed to 'cust_id' in the source file. I fixed it by updating the column name in my code and the pipeline now runs fine.
Agent: Great to hear you nailed it! 🎉

Just a quick recap for future reference:

| Problem | Fix |
|---------|-----|
| Column renamed to `cust_id` | Updated code to use `cust_id` (or renamed the column back to `customer_id` in the source). |

If you run into any other hiccups—whether it’s another column re 

{
  "summary": "The user encountered a KeyError due to a renamed column in a data pipeline. The AI offered a deta

## 3.2 Session 3: planning a trip

In [ ]:
jaipur_session = run_session([
    "Plan a 3-day trip to Jaipur for me. My total budget is 15000 rupees.",
    "Search the web for the best areas to stay in Jaipur on a budget.",
    "I choose to stay near Bani Park. Please remember that I decided on Bani Park.",
])

jaipur_episode = save_episode("Jaipur trip", jaipur_session)
show_episode(jaipur_episode)

Search attempt 1 failed: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/ap
Search attempt 1 failed: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/ap
User : Plan a 3-day trip to Jaipur for me. My total budget is 15000 rupees.
Agent: ## 3‑Day Jaipur Itinerary on ₹15 000  
*(All figures are approximate and based on 2026 prices. Adjust for your exact travel dates and booking window.)*

| Category | Cost (₹) | Notes |
|----------|----------|-------|
| **Travel to Jaipur** (from Delhi by train or bus) | 1 200 | 2‑hour train (Sleeper 

User : Search the web for the best areas to stay in Jaipur on a budget.
Agent: **Best Budget‑Friendly Areas to Stay in Jaipur (2026)**  
*(All information is current as of early 2026. Prices are approximate and can vary with season, booking window, and special events.)*

| Area | Why It’s Budget‑Friendly | Typical Accommodation Types | Approx. Rate (₹/night) | Key Nearby Attra 

User : I choose to s

## 3.3 The store now holds three episodes

In [ ]:
for ep in load_episodes():
    print(f"{ep['timestamp']} | {ep['session']:<15} | {ep['summary'][:90]}...")

2026-11-19 15:26:56 | EV report       | The user requested a concise outline for a research report on electric two‑wheelers in Ind...
2026-11-19 15:27:00 | Pipeline error  | The user encountered a KeyError due to a renamed column in a data pipeline. The AI offered...
2026-11-19 15:28:39 | Jaipur trip     | The user requested a 3‑day Jaipur itinerary within a ₹15,000 budget and later chose to sta...


**What to observe:** the store keeps growing with every session, but **none of it is in the LLM's context window**. From here on, the question is: which episodes should we bring back for the current need?

# 4. Retrieval Type 1: Chronological ("Give Me the Last N Sessions")

We pick episodes **by recency** and put only those into the working memory of a new session.

## 4.1 Helpers

In [ ]:
def episodes_to_message(episodes):
    """Turn retrieved episodes into one system message for the working memory."""
    lines = ["Memory of past sessions with this user:"]
    for ep in episodes:
        events = "; ".join(str(x) for x in ep["key_events"])
        outcomes = "; ".join(str(x) for x in ep["outcomes"])
        lines.append(f"- [{ep['timestamp']}] {ep['summary']} Key events: {events}. Outcomes: {outcomes}.")
    return SystemMessage("\n".join(lines))

def retrieve_recent(n):
    """Chronological retrieval: the last n episodes."""
    all_episodes = load_episodes()
    chosen = all_episodes[-n:]
    print(f"Store has {len(all_episodes)} episodes, retrieved {len(chosen)}: {[e['session'] for e in chosen]}")
    return chosen

## 4.2 A vague question: "continue what we were doing"

This is the exact scenario from the notes. The most recent session is what the user means.

In [ ]:
vague_question = "Can you continue what we were doing last time? Remind me what I decided."

recent_1 = retrieve_recent(1)
ask(vague_question, [episodes_to_message(recent_1)], "2. Chronological (last 1) | vague question", expected="Bani Park")

Store has 3 episodes, retrieved 1: ['Jaipur trip']
[2. Chronological (last 1) | vague question]
Q: Can you continue what we were doing last time? Remind me what I decided.
Tokens sent in first call: 293
A: Sure thing! Here’s a quick recap of what we worked on last time:

| Item | Details |
|------|---------|
| **Trip Length** | 3 days in Jaipur |
| **Budget** | ₹15,000 total |
| **Accommodation** | You chose to stay near **Bani Park** (a convenient area with good access to major attractions and a variety of budget-friendly hotels/guesthouses). |

If you’d like to add more activities, tweak the budget, or need any other details (like transport options, dining suggestions, or a day‑by‑day schedule), just let me know and we can pick up from there!
Contains 'Bani Park'? True



## 4.3 A specific question about an OLDER session

Now the user wants to continue the EV report, which was the **oldest** session. We retrieve the last 2 sessions.

In [ ]:
recent_2 = retrieve_recent(2)
ask(ev_question, [episodes_to_message(recent_2)], "3. Chronological (last 2) | EV question", expected="charging infrastructure")

Store has 3 episodes, retrieved 2: ['Pipeline error', 'Jaipur trip']
[3. Chronological (last 2) | EV question]
Q: Let's continue the electric two-wheeler report. What rule did I give about its content?
Tokens sent in first call: 408
A: I don’t have the rule stored in my memory from this session. Could you remind me what rule you’d like me to follow for the electric two‑wheeler report?
Contains 'charging infrastructure'? False



**What to observe:**
- Chronological retrieval works when the user means "the most recent thing" (4.2).
- It fails when the relevant session is older than the last N (4.3). The EV session is in the store, but we never retrieved it, so the agent can't use it.

# 5. Retrieval Type 2: Semantic ("Find Sessions Similar to This Task")

We turn the **current question into an embedding** and retrieve the episodes whose meaning is closest, no matter how old they are.

## 5.1 Helper

In [ ]:
def retrieve_similar(query, k):
    """Semantic retrieval: the k episodes most similar in meaning to the query."""
    all_episodes = load_episodes()
    q = embedder.encode(query)

    scored = []
    for ep in all_episodes:
        e = np.array(ep["embedding"])
        similarity = float(np.dot(q, e) / (np.linalg.norm(q) * np.linalg.norm(e)))   # cosine similarity
        scored.append((similarity, ep))
    scored.sort(key=lambda pair: pair[0], reverse=True)

    for similarity, ep in scored:
        print(f"  similarity {similarity:.2f} | {ep['session']}")
    chosen = [ep for _, ep in scored[:k]]
    print(f"Store has {len(all_episodes)} episodes, retrieved {len(chosen)}: {[e['session'] for e in chosen]}")
    return chosen

## 5.2 The specific EV question: the one chronological retrieval missed

In [ ]:
similar = retrieve_similar(ev_question, 1)
ask(ev_question, [episodes_to_message(similar)], "4. Semantic (top 1) | EV question", expected="charging infrastructure")

  similarity 0.55 | EV report
  similarity 0.15 | Jaipur trip
  similarity 0.07 | Pipeline error
Store has 3 episodes, retrieved 1: ['EV report']
[4. Semantic (top 1) | EV question]
Q: Let's continue the electric two-wheeler report. What rule did I give about its content?
Tokens sent in first call: 269
A: You asked that the report **exclude any discussion of charging infrastructure**.
Contains 'charging infrastructure'? True



## 5.3 The vague question: the one chronological retrieval handled

In [ ]:
similar = retrieve_similar(vague_question, 1)
ask(vague_question, [episodes_to_message(similar)], "5. Semantic (top 1) | vague question", expected="Bani Park")

  similarity 0.19 | EV report
  similarity 0.15 | Jaipur trip
  similarity 0.10 | Pipeline error
Store has 3 episodes, retrieved 1: ['EV report']
[5. Semantic (top 1) | vague question]
Q: Can you continue what we were doing last time? Remind me what I decided.
Tokens sent in first call: 267
A: Sure! Here’s a quick recap of what we agreed on:

1. **Goal** – Create a concise outline for a research report on electric two‑wheelers in India.  
2. **Structure** – Three‑part outline (e.g., Market Overview, Technology & Adoption, Policy & Future Outlook).  
3. **Rule** – Exclude any discussion of charging infrastructure.

Let me know if you’d like to move forward with the outline, tweak it, or add any other sections.
Contains 'Bani Park'? False



**What to observe:**
- Semantic retrieval finds the old EV session because the question is *about* it (5.2).
- A vague question like "continue what we were doing" says nothing about any topic, so similarity scores are low and close together, and the retrieved episode is not guaranteed to be the one the user means (5.3).
- Each retrieval type has a scenario where it wins and a scenario where it fails. The question decides which one to use.

# 6. Using Stored Key Events and Outcomes

Episodes store more than a summary. Two questions that depend on exactly that.

## 6.1 A key event: a preference the user stated

In [ ]:
event_question = "When we worked on my report, what did I say about how long your answers should be?"

similar = retrieve_similar(event_question, 1)
ask(event_question, [episodes_to_message(similar)], "6. Semantic (top 1) | event question", expected="concise")

  similarity 0.23 | EV report
  similarity 0.15 | Jaipur trip
  similarity 0.08 | Pipeline error
Store has 3 episodes, retrieved 1: ['EV report']
[6. Semantic (top 1) | event question]
Q: When we worked on my report, what did I say about how long your answers should be?
Tokens sent in first call: 269
A: You asked that my responses be **concise**.
Contains 'concise'? True



## 6.2 An outcome: what happened as a result of an action

In [ ]:
outcome_question = "How did I fix the KeyError in my data pipeline?"

similar = retrieve_similar(outcome_question, 1)
ask(outcome_question, [episodes_to_message(similar)], "7. Semantic (top 1) | outcome question", expected="cust_id")

  similarity 0.68 | Pipeline error
  similarity 0.04 | Jaipur trip
  similarity -0.05 | EV report
Store has 3 episodes, retrieved 1: ['Pipeline error']
[7. Semantic (top 1) | outcome question]
Q: How did I fix the KeyError in my data pipeline?
Tokens sent in first call: 260
A: You fixed the KeyError by updating the column name in your pipeline code to match the new name that was used in the source data.

**What happened**

1. **Error** – Your pipeline was throwing `KeyError: 'customer_id'` because it was looking for a column that no longer existed.
2. **Troubleshooting** – You ran through a checklist (checking the schema, logs, and recent changes) and discovered that the column had been renamed.
3. **Rename** – The column that used to be called `customer_id` was renamed to `cust_id` in the upstream data source.
4. **Code update** – You edited the relevant part of your pipeline (e.g., the `select`, `rename`, or `join` statements) to reference `cust_id` instead of `customer_id`.
5. **Res

**What to observe:** facts the user said once, in a past session, now shape the current answer. Nothing was re-typed by the user and only one small episode was sent to the model.

# 7. The Disadvantage: Summaries Lose Detail

An episode is a **compressed** version of the session. The raw transcript is bigger but complete.

We ask a detail-heavy question about the Jaipur session and answer it twice: once with the raw transcript as memory, once with the episode.

In [ ]:
detail_question = "In our Jaipur planning session, list every area that was suggested for staying, as exactly as you can."

# Memory = the full raw transcript of the session (huge, but nothing is lost)
ask(detail_question, jaipur_session, "8. Raw transcript | detail question")

# Memory = only the stored episode (small, but compressed)
ask(detail_question, [episodes_to_message([jaipur_episode])], "9. Episode | detail question")

[8. Raw transcript | detail question]
Q: In our Jaipur planning session, list every area that was suggested for staying, as exactly as you can.
Tokens sent in first call: 3074
A: **Areas Suggested for Staying in Jaipur (exact wording from our discussion)**  

1. **Sindhi Camp**  
2. **Bani Park**  
3. **MI Road (Mahatma Gandhi Road)**  
4. **Panchpura**  
5. **Kishanpura**

[9. Episode | detail question]
Q: In our Jaipur planning session, list every area that was suggested for staying, as exactly as you can.
Tokens sent in first call: 297
A: - Bani Park



**What to observe:**
- Compare the two token counts: the episode is far smaller than the raw transcript.
- Compare the two answers: the transcript can list every area it contains, while the episode only holds what the summary kept. How much is lost depends on what the summariser considered important. Whatever it dropped is gone from the store.
- This is the trade-off behind episodic memory: **small and cheap to retrieve, but not complete**.

# 8. Final Comparison

## 8.1 Real numbers from this notebook

In [ ]:
pd.DataFrame(results).T

,tokens sent,answer has the expected fact
1. No memory | EV question,143,False
2. Chronological (last 1) | vague question,293,True
3. Chronological (last 2) | EV question,408,False
4. Semantic (top 1) | EV question,269,True
5. Semantic (top 1) | vague question,267,False
6. Semantic (top 1) | event question,269,True
7. Semantic (top 1) | outcome question,260,True
8. Raw transcript | detail question,3074,n/a
9. Episode | detail question,297,n/a


## 8.2 Advantages, disadvantages and when to use each

| Approach | Advantage | Disadvantage | Use when |
|---|---|---|---|
| **No memory** | Nothing to build or maintain. | The agent can't continue anything. The user repeats everything. | One-off questions. |
| **Raw transcripts** | Nothing is lost. | Very large, grows with every session, fills the context window. | Almost never. |
| **Episodes + chronological retrieval** | Small. Simple. Ideal for "continue where we left off". | Misses relevant sessions that are older than the last N. | A returning user continuing their latest work. |
| **Episodes + semantic retrieval** | Finds the relevant session even if it is old. | Needs embeddings. Weak on vague questions. Can retrieve the wrong episode. | A task about a specific past topic. |

## 8.3 Key takeaways

1. Episodic memory is **stored outside the LLM**. The context window only ever holds what we retrieve.
2. We store **summaries, key events and outcomes**, not raw transcripts.
3. **Chronological** retrieval answers "what were we just doing?". **Semantic** retrieval answers "when did we work on something like this?".
4. Neither retrieval type wins everywhere, so the question decides which one to use.
5. Compression makes episodes cheap, but whatever the summary drops is lost.